# ChatGLM2-6B 載入實戰：2026 HuggingFace 統一慣例

## 學習目標

完成本 notebook 後，你將能夠：

1. 用 2026 統一慣例（`device_map='auto'`、`torch_dtype=torch.bfloat16`、`use_safetensors=True`）載入大型語言模型。
2. 理解 `bf16` 相對於 `fp16`/`fp32` 的優缺點，以及 `device_map='auto'` 的 CPU/disk offload 語意。
3. 用 `tokenizer.apply_chat_template()` 組裝對話 prompt。
4. 掌握基本量化載入的入口（為下一節 `BitsAndBytesConfig` 4-bit 量化鋪路）。

## 前置條件

- Python 3.10+，CUDA 11.8+ 或 CPU（CPU 推論會慢，但 code 可執行）
- 約 13 GB VRAM（bf16 full precision）；或 8 GB VRAM（搭配 4-bit 量化，見下方替代方案）

## 與相鄰 notebook 的銜接

- 前一節：無（本章入口）
- 後一節：[02-qlora_finetune](../02-qlora_finetune/)（在本節載入的模型基礎上進行 QLoRA 指令微調）
- 量化詳細說明：[03-quantization_deepdive](../03-quantization_deepdive/)（4-bit vs 8-bit vs 16-bit 選擇指引）

In [ ]:
# ── 版本鎖定（執行一次，確保環境一致）──────────────────────────────────────
# 若在乾淨虛擬環境中，取消下列註解安裝依賴
# !pip install -q \
#     "transformers>=4.46" \
#     "accelerate>=1.0" \
#     "bitsandbytes>=0.44" \
#     "safetensors>=0.4" \
#     "torch>=2.4"

import importlib, sys
for pkg in ["transformers", "accelerate", "bitsandbytes", "safetensors", "torch"]:
    try:
        mod = importlib.import_module(pkg)
        print(f"{pkg}: {mod.__version__}")
    except ImportError:
        print(f"{pkg}: NOT INSTALLED")

## 1. 為什麼要統一精度與裝置設定？

### 2026 統一慣例（三個參數一次解決）

| 參數 | 值 | 說明 |
|---|---|---|
| `device_map` | `'auto'` | 自動分配層到可用 GPU，GPU 不夠時 offload 到 CPU，再不夠 offload 到磁碟；無須手動呼叫裝置搬移 |
| `torch_dtype` | `torch.bfloat16` | 比 fp16 數值穩定、比 fp32 省一半 VRAM；bf16 動態範圍遠大於 fp16（指數位元數：bf16=8 vs fp16=5），訓練時數值溢出機率大幅降低 |
| `use_safetensors` | `True` | 使用 `.safetensors` 格式：無 pickle 安全疑慮、支援記憶體映射（載入速度快 2-3x）|

`device_map='auto'` 底層由 `accelerate` 負責，分配邏輯：
1. 盡量放 GPU VRAM
2. VRAM 不足時 offload 到 RAM
3. RAM 不足時 offload 到磁碟（速度最慢，但模型仍可執行）

## 2. 環境設定與模型 ID

2026 做法：優先用 HuggingFace Hub model ID，讓 `transformers` 自動下載並快取到 `~/.cache/huggingface/hub/`。
如果需要離線/私有部署，用環境變數 `HF_MODEL_PATH` 覆蓋即可，不修改 code。

In [ ]:
import os
from pathlib import Path

# 優先從環境變數讀取本地路徑（離線/私有部署場景）
# 若未設定，回退到 Hub model ID（自動下載）
_local_path = os.environ.get("HF_MODEL_PATH", "")
MODEL_ID = _local_path if _local_path else "THUDM/chatglm2-6b"

print(f"Model source: {MODEL_ID}")
print("If you see a Hub ID, the model will be downloaded to ~/.cache/huggingface/hub/")

## 3. 載入 Tokenizer

ChatGLM2 使用自定義的 tokenizer 程式碼，因此 `trust_remote_code=True` 仍然需要。
`AutoTokenizer` 會自動處理快取；若模型已在本地快取，不會重新下載。

> **注意**：`trust_remote_code=True` 意味著你信任來自 HuggingFace Hub 的遠端程式碼會在本機執行。
> 在生產環境，請固定 `revision` 參數（如 `revision='v1.0'`）以鎖定特定版本，避免模型作者更新程式碼後行為改變。

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,   # ChatGLM2 has custom tokenizer code
    # revision='v1.0',        # recommended for production to pin a specific version
)

print(f"Tokenizer type  : {type(tokenizer).__name__}")
print(f"Vocab size      : {tokenizer.vocab_size:,}")
print(f"Model max length: {tokenizer.model_max_length}")
print(f"BOS token       : {tokenizer.bos_token!r}")
print(f"EOS token       : {tokenizer.eos_token!r}")
print(f"PAD token       : {tokenizer.pad_token!r}")

## 4. 載入模型（2026 統一慣例）

### VRAM 需求說明

| 精度 | 約需 VRAM | 推薦硬體 |
|---|---|---|
| bf16（本節預設）| ~13 GB | RTX 3090 / 4090 / A100 |
| 8-bit 量化 | ~8 GB | RTX 3080 / A10 |
| 4-bit 量化（下一節）| ~5 GB | RTX 3060 / T4 |

若 VRAM 不足，請直接跳到本 notebook 底部的「輕量替代：4-bit 量化載入」段落。

In [ ]:
import torch
from transformers import AutoModel

# 2026 unified loading convention
# device_map='auto'    : accelerate automatically distributes layers across available hardware
# torch_dtype=bfloat16 : preferred inference precision (numerically stable + VRAM efficient)
# use_safetensors=True : safe and fast weight format
model = AutoModel.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    use_safetensors=True,
)
model.eval()  # inference mode: disable dropout, no gradient tracking

print(f"Model type   : {type(model).__name__}")
print(f"Torch dtype  : {next(model.parameters()).dtype}")
print(f"Device map   : {model.hf_device_map if hasattr(model, 'hf_device_map') else 'N/A'}")

## 5. 模型架構概覽

輸出模型架構有助於理解層的組成，這也是之後 QLoRA 設定 `target_modules` 的依據。

In [ ]:
# print model architecture summary
print(model)
print()

# count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters     : {total_params / 1e9:.2f}B")
print(f"Trainable parameters : {trainable_params / 1e9:.2f}B")

## 6. 對話推論：apply_chat_template（2026 統一慣例）

`tokenizer.apply_chat_template()` 是 2026 跨模型通用的對話 prompt 組裝方式，讀取 tokenizer 內建的 `chat_template`（Jinja2 格式），保證 prompt 格式與模型訓練時完全一致。

使用方式：

```python
messages = [{"role": "user", "content": "你好"}]
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,  # 在末尾加上 assistant 起始 token
)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
output = model.generate(**inputs, max_new_tokens=256)
response = tokenizer.decode(output[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
```

換模型時只需換 `MODEL_ID`，無需修改任何對話組裝邏輯——這是通往 05-Multimodal 的關鍵橋樑。
多模態訊息（image token、audio token）也可透過相同介面傳遞。

In [ ]:
# 2026 unified chat inference: apply_chat_template
messages = [
    {"role": "user", "content": "用一句話解釋什麼是大語言模型（LLM）。"}
]

# 1. assemble prompt (tokenize=False for easy inspection of the raw format)
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,  # append assistant start marker to trigger generation
)
print("=== Prompt (raw) ===")
print(repr(prompt[:200]), "...")
print()

# 2. encode and run inference
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,       # greedy decode for reproducibility
        eos_token_id=tokenizer.eos_token_id,
    )

# 3. decode only newly generated tokens (strip off the input portion)
new_tokens = output_ids[0][inputs.input_ids.shape[-1]:]
response = tokenizer.decode(new_tokens, skip_special_tokens=True)

print("=== Response ===")
print(response)

### 多輪對話

`apply_chat_template` 天然支援多輪對話，只需把歷史訊息全部放入 `messages` list 即可。
模型不維護內部狀態，每次推論都是全量輸入——這正是 KV Cache 的設計動機（詳見 05 模組）。

In [ ]:
# multi-turn conversation demo
conversation_history: list[dict] = []

def chat_turn(user_message: str) -> str:
    """Single turn of conversation using apply_chat_template."""
    conversation_history.append({"role": "user", "content": user_message})

    prompt = tokenizer.apply_chat_template(
        conversation_history,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0][inputs.input_ids.shape[-1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)

    # append model response to history so the next turn carries this context
    conversation_history.append({"role": "assistant", "content": response})
    return response

# first turn
r1 = chat_turn("量化（Quantization）是什麼？")
print(f"[Turn 1] {r1}\n")

# second turn (with context)
r2 = chat_turn("它和知識蒸餾有什麼不同？")
print(f"[Turn 2] {r2}")

## 7. 輕量替代：4-bit 量化載入（VRAM < 8 GB）

若 VRAM 不足以載入 bf16 full precision 模型，可改用 4-bit 量化。
`BitsAndBytesConfig` 是 2026 設定量化參數的標準入口，應透過 `quantization_config` 參數傳入 `from_pretrained()`。

### 精度比較速查表

| 量化 | VRAM（6B 模型）| 速度 | 精度損失 | 適用場景 |
|---|---|---|---|---|
| bf16（全精度）| ~13 GB | 最快 | 無 | 有足夠 VRAM 的推論/微調 |
| 8-bit | ~8 GB | 略慢 | 極低 | VRAM 受限的推論 |
| 4-bit NF4 | ~5 GB | 較慢 | 低（NF4 優於 FP4）| VRAM 嚴重受限；QLoRA 微調必用 |

`nf4`（NormalFloat4）相對 `fp4` 的優勢：NF4 是針對正態分布權重設計的最優 4-bit 資料型態，資訊損失比 FP4 低約 1/3。
`bnb_4bit_use_double_quant=True`：對量化常數本身再做一次 8-bit 量化，每個參數額外節省 ~0.4 bit，幾乎無精度損失。

In [ ]:
# 4-bit quantized loading (use when VRAM < 8 GB; skip if bf16 model above is already loaded)
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",              # NormalFloat4: lower precision loss than fp4
    bnb_4bit_compute_dtype=torch.bfloat16,  # restore to bf16 during computation
    bnb_4bit_use_double_quant=True,         # double quantization: compress the quant constants too
)

print("BitsAndBytesConfig:")
print(f"  quant_type        : {bnb_config.bnb_4bit_quant_type}")
print(f"  compute_dtype     : {bnb_config.bnb_4bit_compute_dtype}")
print(f"  double_quant      : {bnb_config.bnb_4bit_use_double_quant}")
print()
print("To load the 4-bit model, run:")
print("""
model_4bit = AutoModel.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    device_map='auto',
    quantization_config=bnb_config,
)
""")
print("Note: use_safetensors is not specified with quantization_config;")
print("bitsandbytes handles the format internally after loading.")

### 4-bit 量化載入（實際執行版）

若已有足夠 VRAM，可把上方 `model` 刪除後執行以下 cell，或在獨立核心中執行（避免 OOM）。

In [ ]:
# set IMPORT_4BIT = True to enable 4-bit loading
# ensure VRAM >= 5 GB and the bf16 model above has been released first
IMPORT_4BIT = False

if IMPORT_4BIT:
    import gc
    # release the full-precision model
    del model
    gc.collect()
    torch.cuda.empty_cache()

    model_4bit = AutoModel.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
        device_map="auto",
        quantization_config=bnb_config,
    )
    model_4bit.eval()
    print(f"4-bit model loaded. dtype: {next(model_4bit.parameters()).dtype}")
else:
    print("IMPORT_4BIT=False — skipped. Set to True to load the 4-bit model.")

## 8. Tokenizer 特性探索

了解 tokenizer 的行為對微調至關重要：如何切詞、特殊 token 的 ID 為何、中文字詞的分詞粒度等。

In [ ]:
# tokenizer behaviour exploration
sample_text = "大語言模型（Large Language Model）是基於 Transformer 架構的深度學習模型。"

tokens = tokenizer.tokenize(sample_text)
token_ids = tokenizer.encode(sample_text)

print(f"Input text  : {sample_text}")
print(f"Token count : {len(tokens)}")
print(f"Tokens      : {tokens}")
print(f"Token IDs   : {token_ids}")
print()

# verify encode -> decode roundtrip consistency
decoded = tokenizer.decode(token_ids, skip_special_tokens=True)
print(f"Roundtrip   : {decoded}")
print(f"Lossless    : {sample_text == decoded}")

## 9. 小結與練習

### 本節完成事項

- 以 `device_map='auto'`、`torch_dtype=torch.bfloat16`、`use_safetensors=True` 三個參數完成模型載入，自動處理多 GPU 分層與精度設定。
- 以 HuggingFace Hub model ID 與環境變數 `HF_MODEL_PATH` 取代硬路徑，讓 notebook 在任何環境可執行。
- 以 `apply_chat_template` 組裝對話 prompt，保證格式與模型訓練時一致，且跨模型通用。
- 展示了 `BitsAndBytesConfig` 透過 `quantization_config` 傳入的正確用法，為下一節 QLoRA 微調做準備。

### 練習題

1. **精度對比**：把 `torch_dtype` 改為 `torch.float16`，再次推論同一個問題，比較輸出是否有差異。再試 `torch.float32`，觀察 VRAM 使用量變化（可用 `torch.cuda.memory_allocated()` 量測）。

2. **模型可攜性驗證**：把 `MODEL_ID` 換成 `Qwen/Qwen2-7B-Instruct`（不需 `trust_remote_code`），看看 `apply_chat_template` 是否仍能正確組裝 prompt，對話程式碼是否一行都不需要修改。

3. **Chat Template 探索**：執行 `print(tokenizer.chat_template)` 查看 ChatGLM2 的 Jinja2 模板原始碼，理解 `add_generation_prompt=True` 到底在末尾插入了什麼 token。

4. **量化載入**：把 `IMPORT_4BIT` 設為 `True`，載入 4-bit 量化版本，比較推論輸出與全精度版本的差異，並用 `torch.cuda.memory_allocated()` 確認 VRAM 節省量。

### 下一步

前往 [02-qlora_finetune](../02-qlora_finetune/) 學習如何在本節載入的模型基礎上，用 QLoRA（4-bit + LoRA）進行指令微調，並了解 `prepare_model_for_kbit_training()` 在 PEFT 前的正確使用時機。